In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# V16 GRANDMASTER — SYMMETRIC V2-S (PAPER-SAFE)
# 15 Epochs | Mild Focal | SWA | Full Metrics & Plots
# ============================================================

import os, gc, random, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from PIL import Image, ImageFile
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------------------------------------------
# 0. SETUP
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Running V16 Symmetric V2-S on {DEVICE}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

CONFIG = {
    "ROOT": "/kaggle/input/casia-20-image-tampering-detection-dataset/CASIA2",
    "SIZE": 320,
    "BATCH": 8,
    "EPOCHS": 15,
    "LR": 2e-4,
    "WD": 2e-3,
    "SWA_START": 10,
    "SWA_LR": 1e-4,
    "DEPLOY_THR": 0.35
}

# ------------------------------------------------------------
# 1. FORENSIC LAYERS
# ------------------------------------------------------------
class SRMLayer(nn.Module):
    def __init__(self):
        super().__init__()
        f1=[[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]]
        f2=[[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]]
        f3=[[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]]
        w=np.array([f1,f2,f3])/4.0
        self.register_buffer("weight", torch.FloatTensor(w).unsqueeze(1))

    def forward(self,x):
        return F.conv2d(x.mean(1,keepdim=True), self.weight, padding=2)

class SobelLayer(nn.Module):
    def __init__(self):
        super().__init__()
        sx=torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]],dtype=torch.float32).view(1,1,3,3)
        sy=torch.tensor([[1,2,1],[0,0,0],[-1,-2,-1]],dtype=torch.float32).view(1,1,3,3)
        self.register_buffer("sx",sx)
        self.register_buffer("sy",sy)

    def forward(self,x):
        g=x.mean(1,keepdim=True)
        gx=F.conv2d(g,self.sx,padding=1)
        gy=F.conv2d(g,self.sy,padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-3)

# ------------------------------------------------------------
# 2. FOCAL LOSS (SOFTENED)
# ------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.50, gamma=1.2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        at = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (at * (1 - pt)**self.gamma * ce).mean()

# ------------------------------------------------------------
# 3. MODEL
# ------------------------------------------------------------
class SEBlock(nn.Module):
    def __init__(self,c,r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(c,c//r,bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(c//r,c,bias=False),
            nn.Sigmoid()
        )
    def forward(self,x):
        return x * self.fc(x)

class TripleStreamV16(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

        def backbone(in_ch):
            m = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
            if in_ch != 3:
                old = m.features[0][0]
                new = nn.Conv2d(in_ch, old.out_channels,
                                kernel_size=old.kernel_size,
                                stride=old.stride,
                                padding=old.padding, bias=False)
                new.weight.data = old.weight.sum(1,keepdim=True)/3.0
                m.features[0][0] = new
            m.classifier = nn.Identity()
            return m

        self.rgb = backbone(3)
        self.noise = backbone(3)
        self.edge = backbone(1)
        self.srm = SRMLayer()
        self.sobel = SobelLayer()
        self.se = SEBlock(1280*3)

        self.cls = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(1280*3,1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(1024,2)
        )

    def forward(self,x):
        f = torch.cat([
            self.rgb(x),
            self.noise(self.srm(x)),
            self.edge(self.sobel(x))
        ],1)
        return self.cls(self.se(f))

# ------------------------------------------------------------
# 4. DATA
# ------------------------------------------------------------
class CASIADataset(Dataset):
    def __init__(self,root,tfm):
        self.tfm=tfm
        self.samples=[]
        for r,_,f in os.walk(os.path.join(root,"Au")):
            for i in f:
                if i.lower().endswith((".jpg",".png",".tif")):
                    self.samples.append((os.path.join(r,i),0))
        for r,_,f in os.walk(os.path.join(root,"Tp")):
            for i in f:
                if "_S_" in i or "Tp_S" in i:
                    self.samples.append((os.path.join(r,i),1))
        random.shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self,i):
        p,l=self.samples[i]
        try:
            return self.tfm(Image.open(p).convert("RGB")), l
        except:
            return torch.zeros((3,CONFIG["SIZE"],CONFIG["SIZE"])), l

tfm_train = transforms.Compose([
    transforms.Resize((CONFIG["SIZE"],CONFIG["SIZE"])),
    transforms.RandomResizedCrop(CONFIG["SIZE"], scale=(0.9,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

tfm_val = transforms.Compose([
    transforms.Resize((CONFIG["SIZE"],CONFIG["SIZE"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

ds = CASIADataset(CONFIG["ROOT"], tfm_train)
tr, te = torch.utils.data.random_split(ds, [int(0.8*len(ds)), len(ds)-int(0.8*len(ds))])
te.dataset.tfm = tfm_val

train_ld = DataLoader(tr, CONFIG["BATCH"], shuffle=True, num_workers=2)
test_ld  = DataLoader(te, CONFIG["BATCH"], shuffle=False, num_workers=2)

# ------------------------------------------------------------
# 5. TRAINING
# ------------------------------------------------------------
model = TripleStreamV16().to(DEVICE)
swa_model = AveragedModel(model)

criterion = FocalLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WD"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["SWA_START"])
swa_scheduler = SWALR(optimizer, swa_lr=CONFIG["SWA_LR"])
scaler = GradScaler()

losses = []

print("🔥 Training...")
for ep in range(CONFIG["EPOCHS"]):
    model.train()
    ep_loss = 0
    for x,y in tqdm(train_ld, desc=f"Epoch {ep+1}", leave=False):
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ep_loss += loss.item()

    losses.append(ep_loss / len(train_ld))

    if ep >= CONFIG["SWA_START"]:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

update_bn(train_ld, swa_model, device=DEVICE)

# ------------------------------------------------------------
# 6. EVALUATION + PLOTS
# ------------------------------------------------------------
swa_model.eval()
probs, labels = [], []

with torch.no_grad():
    for x,y in test_ld:
        with autocast():
            out = torch.clamp(swa_model(x.to(DEVICE)), -20, 20)
            p = torch.softmax(out,1)[:,1]
        probs.extend(p.cpu().numpy())
        labels.extend(y.numpy())

probs = np.array(probs)
labels = np.array(labels)
preds = (probs > CONFIG["DEPLOY_THR"]).astype(int)

auc = roc_auc_score(labels, probs)
print(f"\nAUC: {auc:.4f}")
print(classification_report(labels, preds, target_names=["Authentic","Spliced"]))

# Confusion Matrix
cm = confusion_matrix(labels, preds)
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(labels, probs)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve")
plt.show()

# PR Curve
p, r, _ = precision_recall_curve(labels, probs)
plt.figure()
plt.plot(r, p)
plt.title("Precision-Recall Curve")
plt.show()

# Training Loss
plt.figure()
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()
